# 45. CD-OPE-S Gate와 Branch 기여도 분석

`alpha_global`, `alpha_local`이 실제로 사용되었는지 확인하고, branch를 inference-time으로 끄거나 켜서 성능 기여를 분리합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch4_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/4장/ch4_utils.py")) + list(Path.cwd().glob("**/ch4_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "4장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch4_utils import *

paths = find_ch4_paths()
set_korean_font()
set_seed(41)
paths

Chapter4Paths(chapter4_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장'), chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/manifests'), design_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/chapter4_cd_ope_s_architecture_design.md'), validity_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/chapter4_cd_ope_s_design_validity_review.md'))

## 45-1. gate 값 수집

In [2]:
seed_metrics = collect_ch4_cd_ope_metrics()
if seed_metrics.empty:
    raise FileNotFoundError("42번에서 CD-OPE-S 학습 run을 먼저 생성하세요.")
display(seed_metrics[["variant", "model_seed", "red_dice", "seen_color_dice", "alpha_global", "alpha_local", "run_dir"]])

,variant,model_seed,red_dice,seen_color_dice,alpha_global,alpha_local,run_dir
0,g_cd,0,0.530599,0.558469,0.061157,0.000000,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
1,g_cd,1,0.495976,0.621262,0.040685,0.000000,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
2,g_cd,2,0.478978,0.526553,0.056230,0.000000,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
3,gl_cd,0,0.411909,0.560491,0.052356,0.022873,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
4,gl_cd,1,0.382798,0.535433,0.045000,-0.060026,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
5,gl_cd,2,0.488751,0.514625,0.065900,0.026714,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
6,gl_cd_consistency,0,0.503065,0.626133,0.051055,0.025223,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
7,gl_cd_consistency,1,0.486591,0.595692,0.041712,-0.065609,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
8,gl_cd_consistency,2,0.581388,0.619006,0.069178,0.021453,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...
9,gl_cd_consistency_style,0,0.481942,0.578172,0.066665,0.021097,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...


## 45-2. branch ablation 평가

In [3]:
manifests = create_ch4_manifests(max_per_cell=3, seed=41)
TARGET_VARIANT = "gl_cd_consistency_style"
TARGET_SEED = 0
selected = seed_metrics[
    (seed_metrics["variant"] == TARGET_VARIANT)
    & (seed_metrics["model_seed"] == TARGET_SEED)
]
if selected.empty:
    raise FileNotFoundError(f"선택한 run이 없습니다: {TARGET_VARIANT} seed {TARGET_SEED}")
run_dir = Path(selected.iloc[0]["run_dir"])

out_dir = paths.runs_root / "cd_ope_s" / "branch_ablation" / TARGET_VARIANT / f"seed_{TARGET_SEED}"
ablation = evaluate_cd_ope_branch_ablation(
    run_dir,
    manifests["eval_matched_probe"],
    out_dir,
    batch_size=8,
)
display(ablation)

C:\Users\준승\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,mode,mean_dice,seen_color_dice,heldout_color_dice,red_dice,purple_dice,worst_combo_dice,target_fnr
0,full,0.551071,0.591750,0.490052,0.443207,0.536896,0.195665,0.506994
1,rgb_only,0.509136,0.567550,0.421514,0.393534,0.449495,0.119342,0.540647
2,global_only,0.526030,0.570346,0.459556,0.402305,0.516806,0.125541,0.527157
3,local_only,0.524246,0.587227,0.429774,0.400186,0.459362,0.133117,0.523508


## 45-3. 기여도 판정

In [4]:
if "full" in set(ablation["mode"]) and "rgb_only" in set(ablation["mode"]):
    full_red = float(ablation.loc[ablation["mode"] == "full", "red_dice"].iloc[0])
    rgb_red = float(ablation.loc[ablation["mode"] == "rgb_only", "red_dice"].iloc[0])
    print("full - rgb_only red Dice delta:", full_red - rgb_red)

full - rgb_only red Dice delta: 0.04967338880515704
